# Decision Tree - Predicao de resultado de conflito (CS:GO)

Treina um `DecisionTreeClassifier` do PySpark MLlib sobre o dataset tratado gerado em `EDA_Tratamento.ipynb`.
A variavel alvo e `res_conflito` (1 = jogador sobreviveu e matou, 0 = jogador morreu no conflito).

**Estrategia de validacao:** `TrainValidationSplit` com grade pequena de `maxDepth` - leve o suficiente para rodar em WSL2 + Docker com 5 GB de RAM, e academicamente equivalente ao k-fold para demonstrar tuning de hiperparametros.

```
Dataset completo (100%)
|
+-- 80% -> train_df  (entra no TrainValidationSplit)
|         +-- 80% interno -> treino real  (64% do total)
|         +-- 20% interno -> validacao p/ escolher maxDepth (16% do total)
|
+-- 20% -> test_df   (holdout - nunca visto durante o tuning)
```

## Celula 1 - SparkSession e imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

spark = SparkSession.builder.appName("csgo-dt").getOrCreate()
print("Spark:", spark.version)

Spark: 3.5.0


## Celula 2 - Carregar e inspecionar o dataset tratado

In [27]:
df = spark.read.parquet("/home/jovyan/work/data/dataset_tratado")

print("=== Schema ===")
df.printSchema()

print("\n=== Balanceamento de classes ===")
df.groupBy("res_conflito").count().orderBy("res_conflito").show()

print(f"Total de linhas: {df.count()}")

=== Schema ===
root
 |-- tick: integer (nullable = true)
 |-- kills_from_avg: double (nullable = true)
 |-- deaths_from_avg: double (nullable = true)
 |-- distance_closest_enemy: double (nullable = true)
 |-- isBlinded: integer (nullable = true)
 |-- isAirborne: integer (nullable = true)
 |-- isDucking: integer (nullable = true)
 |-- isStanding: integer (nullable = true)
 |-- isScoped: integer (nullable = true)
 |-- isWalking: integer (nullable = true)
 |-- hasHelmet: integer (nullable = true)
 |-- hp: long (nullable = true)
 |-- armor: long (nullable = true)
 |-- equipmentValue: long (nullable = true)
 |-- cash: long (nullable = true)
 |-- total_hp_enemy: long (nullable = true)
 |-- total_hp_team: long (nullable = true)
 |-- num_enemy_alive: long (nullable = true)
 |-- num_team_alive: long (nullable = true)
 |-- enemy_in_range_200: long (nullable = true)
 |-- enemy_in_range_500: long (nullable = true)
 |-- enemy_in_range_1000: long (nullable = true)
 |-- enemy_in_range_2000: long (nul

## Celula 3 - Definir as colunas de feature

As colunas `LEAKAGE_COLS` sao derivadas via `LEAD` (valor do tick seguinte) e **nao devem entrar no modelo** - elas codificam o resultado futuro, o que tornaria a predicao trivial e inutil em producao.

In [28]:
# Colunas que nao sao features
DROP_COLS = [
    "res_conflito",  # label - target do modelo
    "match",         # id de alta cardinalidade, sem valor preditivo direto
    "player",
    "roundNum",
    "tick",
    "kills_from_avg",
    "deaths_from_avg",
    "team_id",
    "cash",
]

# Colunas derivadas de LEAD ("futuro") - data leakage se incluidas
LEAKAGE_COLS = [
    "delta_hp", "delta_armor", "delta_enemy_alive",
    "delta_team_alive", "delta_kills", "performance_score",
    "inimigos_mortos", "delta_rank",
]

# Colunas string que precisam de StringIndexer
STRING_COLS = ["map"]

# Todas as demais colunas numericas/binarias vao direto para o VectorAssembler
numeric_feature_cols = [
    c for c in df.columns
    if c not in DROP_COLS + LEAKAGE_COLS + STRING_COLS
]

print(f"Features numericas ({len(numeric_feature_cols)}):")
for col in numeric_feature_cols:
    print(f"  {col}")

Features numericas (43):
  distance_closest_enemy
  isBlinded
  isAirborne
  isDucking
  isStanding
  isScoped
  isWalking
  hasHelmet
  hp
  armor
  equipmentValue
  total_hp_enemy
  total_hp_team
  num_enemy_alive
  num_team_alive
  enemy_in_range_200
  enemy_in_range_500
  enemy_in_range_1000
  enemy_in_range_2000
  enemy_hp_in_range_500
  enemy_hp_in_range_1000
  enemy_hp_in_range_2000
  enemy_equipment_in_range_500
  enemy_equipment_in_range_1000
  enemy_equipment_in_range_2000
  team_in_range_200
  team_in_range_500
  team_in_range_1000
  equipment_value_team
  equipment_value_enemy
  hp_closest_enemy
  weapon_rifle
  weapon_sniper
  weapon_pistol
  weapon_smg
  weapon_melee
  weapon_utility
  enemy_weapon_rifle
  enemy_weapon_sniper
  enemy_weapon_pistol
  enemy_weapon_smg
  enemy_weapon_melee
  enemy_weapon_utility


## Celula 4 - Pipeline: StringIndexer + VectorAssembler + DecisionTreeClassifier

In [29]:
# 1. Converte 'map' (string) em indice numerico ordenado por frequencia
map_indexer = StringIndexer(
    inputCol="map",
    outputCol="map_idx",
    handleInvalid="keep",   # indice extra para valores novos no conjunto de teste
)

# 2. Junta todas as features em um unico vetor denso chamado 'features'
assembler = VectorAssembler(
    inputCols=numeric_feature_cols + ["map_idx"],
    outputCol="features",
    handleInvalid="skip",   # descarta linhas com NaN residual
)

# 3. Classificador de arvore de decisao - maxDepth sera ajustado pelo TrainValidationSplit
dt = DecisionTreeClassifier(
    labelCol="res_conflito",
    featuresCol="features",
    maxDepth=5,   # valor inicial; a grade vai sobrescrever durante o tuning
    maxBins=32,
    seed=42,
)

# 4. Pipeline: encadeia os 3 estagios garantindo ordem correta em fit e transform
pipeline = Pipeline(stages=[map_indexer, assembler, dt])

print("Pipeline criado com estagios:", [s.__class__.__name__ for s in pipeline.getStages()])

Pipeline criado com estagios: ['StringIndexer', 'VectorAssembler', 'DecisionTreeClassifier']


## Celula 5 - Split holdout (test set isolado)

In [30]:
# Separa 20% como test holdout - esse conjunto nunca sera visto durante o tuning
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Disponivel para tuning : {train_df.count()} linhas")
print(f"Test holdout (isolado) : {test_df.count()} linhas")

Disponivel para tuning : 246966 linhas
Test holdout (isolado) : 61651 linhas


## Celula 6 - ParamGridBuilder + TrainValidationSplit + treinamento

O `TrainValidationSplit` testa cada combinacao de `maxDepth` fazendo um unico split interno (80% treino / 20% validacao), o que e muito mais leve que k-fold e compativel com o ambiente WSL2 + Docker 5 GB.

In [31]:
# Grade de hiperparametros: 3 candidatos de maxDepth
param_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 7])
    .addGrid(dt.maxBins,  [32])       # fixo; aumentar se houver muitas categorias
    .build()
)

# Criterio de selecao do melhor modelo durante o tuning
evaluator = MulticlassClassificationEvaluator(
    labelCol="res_conflito",
    predictionCol="prediction",
    metricName="f1",
)

# TrainValidationSplit: para cada combinacao de params, treina em 80% e valida em 20%
tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,     # 80% treino interno, 20% validacao interna
    seed=42,
    parallelism=1,      # sequencial - evita pico de memoria no WSL2
)

print("Iniciando TrainValidationSplit com", len(param_grid), "combinacoes...")
tvs_model = tvs.fit(train_df)

# Exibir qual maxDepth foi escolhido
best_dt = tvs_model.bestModel.stages[-1]
print(f"\nMelhor maxDepth : {best_dt.depth}")
print(f"Numero de nos   : {best_dt.numNodes}")

# F1 de cada candidato durante a validacao interna
print("\nF1 por candidato (validacao interna):")
for params, metric in zip(param_grid, tvs_model.validationMetrics):
    depth = params[dt.maxDepth]
    print(f"  maxDepth={depth}  ->  F1={metric:.4f}")

Iniciando TrainValidationSplit com 3 combinacoes...

Melhor maxDepth : 7
Numero de nos   : 147

F1 por candidato (validacao interna):
  maxDepth=3  ->  F1=0.6244
  maxDepth=5  ->  F1=0.6372
  maxDepth=7  ->  F1=0.6419


## Celula 7 - Predicoes e avaliacao no test holdout

In [32]:
# Aplica o melhor pipeline ao test holdout (nunca visto durante o tuning)
predictions = tvs_model.transform(test_df)

# Accuracy
acc_eval = MulticlassClassificationEvaluator(
    labelCol="res_conflito",
    predictionCol="prediction",
    metricName="accuracy",
)
print(f"Accuracy : {acc_eval.evaluate(predictions):.4f}")

# F1-score
f1_eval = MulticlassClassificationEvaluator(
    labelCol="res_conflito",
    predictionCol="prediction",
    metricName="f1",
)
print(f"F1-score : {f1_eval.evaluate(predictions):.4f}")

# AUC-ROC
auc_eval = BinaryClassificationEvaluator(
    labelCol="res_conflito",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
print(f"AUC-ROC  : {auc_eval.evaluate(predictions):.4f}")

Accuracy : 0.6437
F1-score : 0.6440
AUC-ROC  : 0.5827


## Celula 8 - Matriz de confusao

In [33]:
print("Matriz de confusao (real x previsto):")
print("  res_conflito=0 -> jogador morreu")
print("  res_conflito=1 -> jogador sobreviveu e matou\n")

predictions.groupBy("res_conflito", "prediction") \
    .count() \
    .orderBy("res_conflito", "prediction") \
    .show()

Matriz de confusao (real x previsto):
  res_conflito=0 -> jogador morreu
  res_conflito=1 -> jogador sobreviveu e matou

+------------+----------+-----+
|res_conflito|prediction|count|
+------------+----------+-----+
|           0|       0.0|20501|
|           0|       1.0|11997|
|           1|       0.0| 9969|
|           1|       1.0|19184|
+------------+----------+-----+



## Celula 9 - Feature importance

Score baseado em **reducao de impureza de Gini** - quanto maior, mais aquela feature contribuiu para separar as classes na arvore.

In [34]:
best_pipeline_model = tvs_model.bestModel
dt_model = best_pipeline_model.stages[-1]  # DecisionTreeClassificationModel

feature_names = numeric_feature_cols + ["map_idx"]
importances = dt_model.featureImportances

print(f"{'Feature':<45} {'Importancia':>12}")
print("-" * 59)

for name, score in sorted(
    zip(feature_names, importances.toArray()),
    key=lambda x: -x[1],
):
    if score > 0:
        print(f"{name:<45} {score:>12.4f}")

Feature                                        Importancia
-----------------------------------------------------------
hp                                                  0.4556
enemy_hp_in_range_1000                              0.1370
enemy_hp_in_range_2000                              0.0932
equipmentValue                                      0.0775
equipment_value_enemy                               0.0694
enemy_equipment_in_range_2000                       0.0499
equipment_value_team                                0.0444
enemy_equipment_in_range_1000                       0.0142
total_hp_enemy                                      0.0109
total_hp_team                                       0.0101
hp_closest_enemy                                    0.0096
isScoped                                            0.0083
team_in_range_1000                                  0.0049
distance_closest_enemy                              0.0044
isWalking                                           0.0